House Price Predictor — Raw CSV → Trained Model + Evaluation
This project builds a complete machine learning pipeline for predicting house prices using a CSV dataset.
It includes:


Loading raw CSV data


Data cleaning


Feature preprocessing


Train/Test split


Model training


Evaluation metrics


Residual analysis


Actual vs Predicted visualization


Feature coefficient interpretation

In [ ]:
# ============================================
# HOUSE PRICE PREDICTION PROJECT
# Raw CSV -> Trained Model + Evaluation
# ============================================

# -------------------------------
# IMPORT LIBRARIES
# -------------------------------

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# -------------------------------
# LOAD DATA
# -------------------------------

# Replace with your file name
df = pd.read_csv("house_prices.csv")

print("Dataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
print(df.head())

# -------------------------------
# TARGET VARIABLE
# -------------------------------

target = "Price"

X = df.drop(columns=[target])
y = df[target]

# -------------------------------
# IDENTIFY COLUMN TYPES
# -------------------------------

numeric_features = X.select_dtypes(
    include=['int64', 'float64']
).columns

categorical_features = X.select_dtypes(
    include=['object']
).columns

print("\nNumeric Features:")
print(list(numeric_features))

print("\nCategorical Features:")
print(list(categorical_features))

# -------------------------------
# NUMERIC PIPELINE
# -------------------------------

numeric_transformer = Pipeline(steps=[

    ('imputer', SimpleImputer(strategy='median')),

    ('scaler', StandardScaler())
])

# -------------------------------
# CATEGORICAL PIPELINE
# -------------------------------

categorical_transformer = Pipeline(steps=[

    ('imputer', SimpleImputer(strategy='most_frequent')),

    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# -------------------------------
# PREPROCESSOR
# -------------------------------

preprocessor = ColumnTransformer(

    transformers=[

        ('num', numeric_transformer, numeric_features),

        ('cat', categorical_transformer, categorical_features)
    ]
)

# -------------------------------
# MODEL PIPELINE
# -------------------------------

model = Pipeline(steps=[

    ('preprocessor', preprocessor),

    ('regressor', LinearRegression())
])

# -------------------------------
# TRAIN TEST SPLIT
# -------------------------------

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42
)

# -------------------------------
# TRAIN MODEL
# -------------------------------

model.fit(X_train, y_train)

print("\nModel Training Completed")

# -------------------------------
# PREDICTIONS
# -------------------------------

y_pred = model.predict(X_test)

# -------------------------------
# EVALUATION METRICS
# -------------------------------

mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print("\n===== MODEL EVALUATION =====")

print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

# ============================================
# VISUALIZATION 1:
# ACTUAL VS PREDICTED
# ============================================

plt.figure(figsize=(8,6))

plt.scatter(y_test, y_pred)

plt.xlabel("Actual Prices")
plt.ylabel("Predicted Prices")

plt.title("Actual vs Predicted House Prices")

# Perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())

plt.plot(
    [min_val, max_val],
    [min_val, max_val]
)

plt.show()

# ============================================
# VISUALIZATION 2:
# RESIDUAL PLOT
# ============================================

residuals = y_test - y_pred

plt.figure(figsize=(8,6))

plt.scatter(y_pred, residuals)

plt.axhline(y=0)

plt.xlabel("Predicted Prices")
plt.ylabel("Residuals")

plt.title("Residual Plot")

plt.show()

# ============================================
# FEATURE COEFFICIENTS
# ============================================

# Get feature names after preprocessing
ohe = model.named_steps['preprocessor'] \
          .named_transformers_['cat'] \
          .named_steps['onehot']

encoded_cat_features = ohe.get_feature_names_out(categorical_features)

all_features = np.concatenate([
    numeric_features,
    encoded_cat_features
])

# Get coefficients
coefficients = model.named_steps['regressor'].coef_

coef_df = pd.DataFrame({

    'Feature': all_features,
    'Coefficient': coefficients
})

# Sort by impact
coef_df = coef_df.sort_values(
    by='Coefficient',
    ascending=False
)

print("\nTop Positive Features:")
print(coef_df.head(10))

print("\nTop Negative Features:")
print(coef_df.tail(10))

# ============================================
# COEFFICIENT VISUALIZATION
# ============================================

top_features = pd.concat([

    coef_df.head(10),

    coef_df.tail(10)
])

plt.figure(figsize=(10,8))

plt.barh(
    top_features['Feature'],
    top_features['Coefficient']
)

plt.xlabel("Coefficient Value")
plt.ylabel("Feature")

plt.title("Top Feature Coefficients")

plt.show()

# ============================================
# SAVE MODEL (OPTIONAL)
# ============================================

import joblib

joblib.dump(model, "house_price_model.pkl")

print("\nModel Saved Successfully")

What Each Graph Tells
1. Actual vs Predicted Plot

Purpose:
Checks how close predictions are to real house prices.

Interpretation:

Points close to diagonal line → good predictions
Points far away → prediction errors
Tight cluster → strong model
2. Residual Plot

Residual = Actual − Predicted

Purpose:
Checks model mistakes.

Interpretation:

Random scatter around zero → good model
Clear pattern/curve → model missing relationships
Large spread → unstable predictions
3. Feature Coefficient Plot

Purpose:
Shows which features influence price most.

Interpretation:

Positive coefficient → increases price
Negative coefficient → decreases price
Large magnitude → stronger impact

Example:

Square footage → positive impact
Old house age → negative impact
Metrics Explanation
MAE

Average prediction error.

Smaller is better.

RMSE

Punishes large mistakes heavily.

Useful for house price prediction because large pricing mistakes matter.

R² Score

Measures how much variance model explains.

R2=1−SStot​SSres​​
	

Interpretation:

1.0 → perfect predictions
0.8 → strong model
0.5 → moderate
0 → poor model
Business Story You Can Present

“We trained a machine learning model that learns patterns from historical house data such as area, bedrooms, location, and amenities.

The model predicts future house prices and evaluates its own performance using error metrics.

We validated the predictions using residual analysis and actual vs predicted comparisons.

Finally, we identified the most influential features affecting house prices through coefficient analysis.”